<a href="https://colab.research.google.com/github/mmuputisi/Adv-Py_Data_Analysis/blob/main/Nigeria%20public%20health%20supply%20chain%20optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# Supply Chain Optimizer - Final Fixed Version
# =========================

# --- 0️⃣ Install packages in the correct order with compatible versions ---
import sys
print("Setting up environment...")

# First, import getpass for API key input
import getpass

# For OR-Tools, we need a newer protobuf, but TensorFlow needs older
# Strategy: Don't force a specific protobuf version - let pip resolve
print("\nInstalling core packages...")
!pip install --quiet pandas numpy matplotlib seaborn tqdm

# Install OR-Tools first (it will bring its own protobuf)
print("Installing OR-Tools...")
!pip install --quiet ortools

# Now install other packages that might conflict
print("Installing visualization packages...")
!pip install --quiet plotly folium contextily

# Install geospatial packages
print("Installing geospatial packages...")
!pip install --quiet geopandas shapely

# Install machine learning
print("Installing scikit-learn...")
!pip install --quiet scikit-learn

# Install OpenRouteService last
print("Installing OpenRouteService...")
!pip install --quiet openrouteservice

print("\n✅ Package installation complete!")

# --- 1️⃣ Now import everything ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import geopandas as gpd
from shapely.geometry import Point
from sklearn.cluster import KMeans
import contextily as ctx
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timezone
import pickle
import random
import folium
from folium.plugins import MarkerCluster, HeatMap
import warnings
import getpass
from multiprocessing import cpu_count, Pool
warnings.filterwarnings('ignore')

# ORS imports
try:
    import openrouteservice
    from openrouteservice.exceptions import ApiError
    ORS_AVAILABLE = True
    print("✅ OpenRouteService imported successfully")
except ImportError:
    print("⚠️ OpenRouteService not available; will fallback to Haversine distances.")
    ORS_AVAILABLE = False
    openrouteservice = None

# OR-Tools import
try:
    from ortools.constraint_solver import routing_enums_pb2, pywrapcp
    ORTOOLS_AVAILABLE = True
    print("✅ OR-Tools imported successfully")
except ImportError:
    print("⚠️ OR-Tools not available; routing optimization will be skipped.")
    ORTOOLS_AVAILABLE = False

print("✅ All imports complete!")

# --- 2️⃣ Check protobuf version ---
import google.protobuf
print(f"\n📦 Protobuf version: {google.protobuf.__version__}")
print("Note: Some warnings about incompatible versions are expected and usually harmless.")
print("The packages will still work as long as core functionality isn't broken.\n")

# --- 3️⃣ Secure API Key Handling ---
from google.colab import userdata

def get_api_key():
    """Safely get API key from Colab secrets or user input"""
    try:
        # Try to get from Colab secrets
        ORS_API_KEY = userdata.get('ORS_API_KEY')
        if ORS_API_KEY:
            print("✅ Using API key from Colab secrets")
            return ORS_API_KEY
    except:
        pass

    # Fall back to user input (hidden)
    print("⚠️  No API key found in secrets.")
    ORS_API_KEY = getpass.getpass("Enter your OpenRouteService API key (input hidden): ")
    return ORS_API_KEY

ORS_API_KEY = get_api_key()

# --- 4️⃣ Mount Google Drive safely ---
from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
    print("✅ Google Drive mounted")
else:
    print("✅ Google Drive already mounted")

# --- 5️⃣ File paths with error handling ---
EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/health facilities list.xlsx"
CACHE_PATH = "/content/drive/MyDrive/supply_chain_cache/ors_cache.pkl"
OUTPUT_FOLDER = "/content/drive/MyDrive/supply_chain_outputs/"

try:
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    print(f"✅ Output folder: {OUTPUT_FOLDER}")
except Exception as e:
    print(f"⚠️  Could not create output folder: {e}")
    OUTPUT_FOLDER = "./outputs/"
    CACHE_PATH = "./ors_cache.pkl"
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

if not os.path.exists(EXCEL_PATH):
    raise FileNotFoundError(f"❌ Excel file not found at {EXCEL_PATH}")

# --- 6️⃣ Load facilities with error handling ---
print("\n📊 Loading facilities data...")
try:
    df_facilities = pd.read_excel(EXCEL_PATH, header=1)
    df_facilities.columns = df_facilities.iloc[0]
    df_facilities = df_facilities.drop(0).reset_index(drop=True)
    df_facilities = df_facilities.iloc[:, 3:]  # Drop first 3 empty columns
    df_facilities.columns = [str(col).strip() for col in df_facilities.columns]

    # Handle different possible column names
    lat_col = next((col for col in df_facilities.columns if 'lat' in col.lower()), 'Latitude')
    lon_col = next((col for col in df_facilities.columns if 'lon' in col.lower() or 'long' in col.lower()), 'Longitude')

    df_facilities['longitude'] = pd.to_numeric(df_facilities[lon_col], errors='coerce')
    df_facilities['latitude'] = pd.to_numeric(df_facilities[lat_col], errors='coerce')
    df_facilities['frequency'] = pd.to_numeric(df_facilities.get('Frequency', 1), errors='coerce').fillna(1)
    df_facilities = df_facilities.dropna(subset=['longitude','latitude'])

    # Standardize column names
    df_facilities.rename(columns={
        'State':'state',
        'Facility name':'facility_name',
        'Facility type':'facility_type'
    }, inplace=True)

    print(f"✅ Loaded {len(df_facilities)} facilities across {df_facilities['state'].nunique()} states")
    print(f"📈 Facility types: {df_facilities['facility_type'].value_counts().to_dict() if 'facility_type' in df_facilities.columns else 'N/A'}")

except Exception as e:
    print(f"❌ Error loading facilities: {e}")
    raise

# --- 7️⃣ Load city/capital data ---
cities_data = {
    'State': ['Abia', 'Adamawa', 'Akwa-Ibom', 'Anambra', 'Bauchi', 'Bayelsa', 'Benue', 'Borno',
              'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti', 'Enugu', 'Gombe', 'Imo', 'Jigawa',
              'Kaduna', 'Kano', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger',
              'Ogun', 'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe',
              'Zamfara', 'FCT'],
    'Capital': ['Umuahia', 'Yola', 'Uyo', 'Awka', 'Bauchi', 'Yenagoa', 'Makurdi', 'Maiduguri',
                'Calabar', 'Asaba', 'Abakaliki', 'Benin City', 'Ado-Ekiti', 'Enugu', 'Gombe',
                'Owerri', 'Dutse', 'Kaduna', 'Kano', 'Katsina', 'Birnin Kebbi', 'Lokoja', 'Ilorin',
                'Ikeja', 'Lafia', 'Minna', 'Abeokuta', 'Akure', 'Osogbo', 'Ibadan', 'Jos',
                'Port Harcourt', 'Sokoto', 'Jalingo', 'Damaturu', 'Gusau', 'Abuja'],
    'Lat': [5.52491, 9.20839, 5.05127, 6.21269, 10.31032, 4.92675, 7.73375, 11.84692, 4.95893,
            6.19824, 6.32485, 6.33815, 7.62329, 6.44132, 10.28969, 5.48363, 11.75618, 10.52641,
            12.00012, 12.99082, 12.45389, 7.79688, 8.49664, 6.59651, 8.4939, 9.61524, 7.15571,
            7.25256, 7.77104, 7.37756, 9.92849, 4.77742, 13.06269, 8.89367, 11.74697, 12.17024,
            9.05785],
    'Long': [7.49461, 12.48146, 7.9335, 7.07199, 9.84388, 6.26764, 8.52139, 13.15712, 8.32695,
             6.73187, 8.11368, 5.62575, 5.22087, 7.49883, 11.16729, 7.03325, 9.33896, 7.43879,
             8.51672, 7.60177, 4.1975, 6.74048, 4.54214, 3.34205, 8.51532, 6.54776, 3.34509,
             5.19312, 4.55698, 3.90591, 8.89212, 7.0134, 5.24322, 11.3596, 11.96083, 6.66412,
             7.49508]
}
df_cities = pd.DataFrame(cities_data)
df_cities.rename(columns={'Lat':'latitude','Long':'longitude','Capital':'city','State':'state'}, inplace=True)
print(f"✅ Loaded {len(df_cities)} state capitals")

# --- 8️⃣ Haversine distance fallback ---
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two points"""
    R = 6371  # Earth's radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# --- 9️⃣ Enhanced ORS Distance with caching ---
class ORSDistanceCache:
    def __init__(self, api_key=None, cache_path=CACHE_PATH):
        self.client = openrouteservice.Client(key=api_key) if api_key and ORS_AVAILABLE else None
        self.cache_path = cache_path
        self.cache = self._load_cache()
        self.api_calls = 0
        self.cache_hits = 0

    def _load_cache(self):
        """Load cache from disk"""
        if os.path.exists(self.cache_path):
            try:
                with open(self.cache_path, 'rb') as f:
                    cache = pickle.load(f)
                    print(f"✅ Loaded {len(cache)} cached distances")
                    return cache
            except Exception as e:
                print(f"⚠️  Could not load cache: {e}")
        return {}

    def save_cache(self):
        """Save cache to disk"""
        try:
            with open(self.cache_path, 'wb') as f:
                pickle.dump(self.cache, f)
            print(f"✅ Saved {len(self.cache)} distances to cache")
        except Exception as e:
            print(f"⚠️  Could not save cache: {e}")

    def get_distance(self, from_coord, to_coord, use_ors=True):
        """Get distance with caching and fallback"""
        # Normalize coordinates to ensure consistent keys
        from_key = (round(from_coord[0], 6), round(from_coord[1], 6))
        to_key = (round(to_coord[0], 6), round(to_coord[1], 6))
        k = (from_key, to_key)

        # Check cache
        if k in self.cache:
            self.cache_hits += 1
            return self.cache[k]

        # Calculate distance
        if use_ors and self.client:
            try:
                self.api_calls += 1
                res = self.client.directions(
                    [from_coord, to_coord],
                    profile='driving-car',
                    format='json'
                )
                dist = res['routes'][0]['summary']['distance'] / 1000  # km
            except Exception as e:
                # Fallback to haversine
                dist = haversine_distance(
                    from_coord[1], from_coord[0],
                    to_coord[1], to_coord[0]
                )
        else:
            dist = haversine_distance(
                from_coord[1], from_coord[0],
                to_coord[1], to_coord[0]
            )

        # Cache result
        self.cache[k] = dist
        return dist

    def get_stats(self):
        """Get cache statistics"""
        total = self.api_calls + self.cache_hits
        hit_rate = (self.cache_hits / total * 100) if total > 0 else 0
        return {
            'cache_size': len(self.cache),
            'api_calls': self.api_calls,
            'cache_hits': self.cache_hits,
            'hit_rate': hit_rate
        }

# Initialize cache
ors_cache = ORSDistanceCache(api_key=ORS_API_KEY)

# --- 1️⃣0️⃣ Parallel distance calculation ---
def calculate_distances_parallel(args):
    """Helper function for parallel distance calculation"""
    f, warehouses, ors_cache = args
    min_dist = float('inf')
    nearest_wh = None

    for _, w in warehouses.iterrows():
        dist = ors_cache.get_distance(
            (f['longitude'], f['latitude']),
            (w['longitude'], w['latitude'])
        )
        if dist < min_dist:
            min_dist = dist
            nearest_wh = w['city']

    return {
        'warehouse_city': nearest_wh,
        'distance_km': min_dist,
        'transport_cost': min_dist * 0.5  # Base cost per km
    }

def assign_facilities_parallel(facilities, warehouses, ors_cache=None, use_parallel=True):
    """Assign facilities to nearest warehouses with parallel processing"""
    try:
        if use_parallel and cpu_count() > 1:
            print(f"🔄 Using parallel processing with {cpu_count()} cores...")

            # Prepare arguments
            args_list = [(row, warehouses, ors_cache) for _, row in facilities.iterrows()]

            # Process in parallel
            with Pool(processes=min(cpu_count(), 8)) as pool:
                results = list(tqdm(
                    pool.imap(calculate_distances_parallel, args_list),
                    total=len(args_list),
                    desc="Calculating distances"
                ))
        else:
            # Sequential processing
            print("🔄 Using sequential processing...")
            results = []
            for _, f in tqdm(facilities.iterrows(), total=len(facilities), desc="Calculating distances"):
                min_dist = float('inf')
                nearest_wh = None
                for _, w in warehouses.iterrows():
                    dist = ors_cache.get_distance(
                        (f['longitude'], f['latitude']),
                        (w['longitude'], w['latitude'])
                    ) if ors_cache else haversine_distance(
                        f['latitude'], f['longitude'],
                        w['latitude'], w['longitude']
                    )
                    if dist < min_dist:
                        min_dist = dist
                        nearest_wh = w['city']
                results.append({
                    'warehouse_city': nearest_wh,
                    'distance_km': min_dist,
                    'transport_cost': min_dist * 0.5
                })
    except Exception as e:
        print(f"⚠️ Parallel processing failed: {e}")
        print("Falling back to sequential processing...")

        # Sequential fallback
        results = []
        for _, f in tqdm(facilities.iterrows(), total=len(facilities), desc="Calculating distances"):
            min_dist = float('inf')
            nearest_wh = None
            for _, w in warehouses.iterrows():
                dist = ors_cache.get_distance(
                    (f['longitude'], f['latitude']),
                    (w['longitude'], w['latitude'])
                ) if ors_cache else haversine_distance(
                    f['latitude'], f['longitude'],
                    w['latitude'], w['longitude']
                )
                if dist < min_dist:
                    min_dist = dist
                    nearest_wh = w['city']
            results.append({
                'warehouse_city': nearest_wh,
                'distance_km': min_dist,
                'transport_cost': min_dist * 0.5
            })

    # Combine results with facilities
    return pd.concat([facilities.reset_index(drop=True), pd.DataFrame(results)], axis=1)

# --- 1️⃣1️⃣ Enhanced cost analysis ---
def calculate_detailed_costs(scenario_data, fuel_cost_per_km=0.5, vehicle_capacity=1000,
                            driver_cost_per_hour=10, avg_speed=60, maintenance_per_km=0.1):
    """Calculate detailed operational costs"""
    assignments = scenario_data['assignments']
    total_distance = assignments['distance_km'].sum()

    # Basic costs
    fuel_cost = total_distance * fuel_cost_per_km
    maintenance_cost = total_distance * maintenance_per_km

    # Labor costs (assuming 8-hour shifts, 60 km/h average)
    total_hours = total_distance / avg_speed
    driver_cost = total_hours * driver_cost_per_hour

    # Number of trips based on demand
    if 'frequency' in assignments.columns:
        total_demand = assignments['frequency'].sum()
    else:
        total_demand = len(assignments)

    num_trips = np.ceil(total_demand / vehicle_capacity)

    # Per-warehouse breakdown
    warehouse_costs = {}
    for warehouse in scenario_data['warehouses']:
        wh_assignments = assignments[assignments['warehouse_city'] == warehouse]
        wh_distance = wh_assignments['distance_km'].sum()
        wh_demand = wh_assignments['frequency'].sum() if 'frequency' in wh_assignments.columns else len(wh_assignments)

        warehouse_costs[warehouse] = {
            'facilities_served': len(wh_assignments),
            'total_demand': wh_demand,
            'total_distance': wh_distance,
            'fuel_cost': wh_distance * fuel_cost_per_km,
            'driver_cost': (wh_distance / avg_speed) * driver_cost_per_hour,
            'trips_needed': np.ceil(wh_demand / vehicle_capacity)
        }

    return {
        'summary': {
            'fuel_cost': fuel_cost,
            'driver_cost': driver_cost,
            'maintenance_cost': maintenance_cost,
            'num_trips': num_trips,
            'total_cost': fuel_cost + driver_cost + maintenance_cost,
            'cost_per_km': (fuel_cost + driver_cost + maintenance_cost) / total_distance if total_distance > 0 else 0
        },
        'warehouse_breakdown': warehouse_costs
    }

# --- 1️⃣2️⃣ Coverage analysis ---
def analyze_coverage(scenarios, thresholds=[50, 100, 200, 500]):
    """Analyze coverage at different distance thresholds"""
    coverage_stats = {}

    for n_wh, data in scenarios.items():
        coverage = {}
        for threshold in thresholds:
            covered = (data['assignments']['distance_km'] <= threshold).sum()
            coverage[f'≤{threshold}km'] = {
                'count': covered,
                'percentage': (covered / len(data['assignments'])) * 100
            }

        # Additional metrics
        coverage_stats[n_wh] = {
            'coverage_by_threshold': coverage,
            'max_distance': data['max_dist'],
            'avg_distance': data['avg_dist'],
            'std_distance': data['assignments']['distance_km'].std()
        }

    return coverage_stats

# --- 1️⃣3️⃣ OR-Tools vehicle routing optimization ---
if ORTOOLS_AVAILABLE:
    def optimize_routes(warehouse, facilities, num_vehicles=5, ors_cache=None):
        """Optimize delivery routes from a warehouse using OR-Tools"""
        print(f"🔄 Optimizing routes for {warehouse['city']}...")

        # Prepare facilities list (including warehouse as node 0)
        all_points = [warehouse] + facilities.to_dict('records')

        # Create distance matrix
        n = len(all_points)
        distance_matrix = np.zeros((n, n))

        for i in tqdm(range(n), desc="Building distance matrix"):
            for j in range(n):
                if i != j:
                    dist = ors_cache.get_distance(
                        (all_points[i]['longitude'], all_points[i]['latitude']),
                        (all_points[j]['longitude'], all_points[j]['latitude'])
                    )
                    distance_matrix[i][j] = int(dist * 1000)  # Convert to meters

        # Create routing model
        manager = pywrapcp.RoutingIndexManager(n, num_vehicles, 0)
        routing = pywrapcp.RoutingModel(manager)

        def distance_callback(from_index, to_index):
            from_node = manager.IndexToNode(from_index)
            to_node = manager.IndexToNode(to_index)
            return int(distance_matrix[from_node][to_node])

        transit_callback_index = routing.RegisterTransitCallback(distance_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

        # Add distance constraint
        dimension_name = 'Distance'
        routing.AddDimension(
            transit_callback_index,
            0,  # no slack
            300000,  # vehicle maximum travel distance (300 km)
            True,  # start cumul to zero
            dimension_name
        )
        distance_dimension = routing.GetDimensionOrDie(dimension_name)
        distance_dimension.SetGlobalSpanCostCoefficient(100)

        # Set search parameters
        search_parameters = pywrapcp.DefaultRoutingSearchParameters()
        search_parameters.first_solution_strategy = (
            routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
        )
        search_parameters.local_search_metaheuristic = (
            routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
        )
        search_parameters.time_limit.seconds = 30

        # Solve
        solution = routing.SolveWithParameters(search_parameters)

        if solution:
            routes = []
            for vehicle_id in range(num_vehicles):
                index = routing.Start(vehicle_id)
                route = []
                route_distance = 0

                while not routing.IsEnd(index):
                    node_index = manager.IndexToNode(index)
                    if node_index > 0:  # Skip warehouse (node 0)
                        route.append(all_points[node_index])

                    previous_index = index
                    index = solution.Value(routing.NextVar(index))
                    route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

                if route:  # Only add non-empty routes
                    routes.append({
                        'vehicle_id': vehicle_id,
                        'stops': len(route),
                        'distance_km': route_distance / 1000,
                        'facilities': route
                    })

            return routes
        else:
            print(f"⚠️  No solution found for {warehouse['city']}")
            return None
else:
    def optimize_routes(warehouse, facilities, num_vehicles=5, ors_cache=None):
        print("⚠️  OR-Tools not available. Skipping route optimization.")
        return None

# --- 1️⃣4️⃣ Interactive map creation ---
def create_interactive_map(scenario_data, n_wh, output_path=None):
    """Create an interactive Folium map"""
    print(f"🗺️  Creating interactive map for {n_wh} warehouses...")

    # Calculate center
    center_lat = scenario_data['assignments']['latitude'].mean()
    center_lon = scenario_data['assignments']['longitude'].mean()

    # Create map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=6)

    # Get warehouse locations
    wh_locations = df_cities[df_cities['city'].isin(scenario_data['warehouses'])]

    # Color palette for different warehouses
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred',
              'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue',
              'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen',
              'gray', 'black', 'lightgray']

    # Add facility markers by warehouse
    for i, warehouse in enumerate(scenario_data['warehouses']):
        color = colors[i % len(colors)]

        # Get facilities for this warehouse
        wh_facilities = scenario_data['assignments'][
            scenario_data['assignments']['warehouse_city'] == warehouse
        ]

        # Create feature group for this warehouse
        fg = folium.FeatureGroup(name=f"Served by {warehouse}")

        # Add facilities
        for _, facility in wh_facilities.iterrows():
            popup_text = f"""
            <b>{facility.get('facility_name', 'Unknown')}</b><br>
            Type: {facility.get('facility_type', 'Unknown')}<br>
            Distance: {facility['distance_km']:.1f} km<br>
            Cost: ${facility['transport_cost']:.2f}
            """

            folium.CircleMarker(
                [facility['latitude'], facility['longitude']],
                radius=5,
                color=color,
                fill=True,
                popup=popup_text,
                tooltip=facility.get('facility_name', 'Facility')
            ).add_to(fg)

        fg.add_to(m)

    # Add warehouse markers
    wh_fg = folium.FeatureGroup(name="Warehouses", show=True)
    for _, warehouse in wh_locations.iterrows():
        # Count facilities served
        n_served = len(scenario_data['assignments'][
            scenario_data['assignments']['warehouse_city'] == warehouse['city']
        ])

        popup_text = f"""
        <b>🏢 {warehouse['city']}</b><br>
        Facilities served: {n_served}<br>
        State: {warehouse['state']}
        """

        folium.Marker(
            [warehouse['latitude'], warehouse['longitude']],
            popup=popup_text,
            tooltip=f"🏢 {warehouse['city']}",
            icon=folium.Icon(color='black', icon='info-sign')
        ).add_to(wh_fg)

    wh_fg.add_to(m)

    # Add layer control
    folium.LayerControl().add_to(m)

    # Save if path provided
    if output_path:
        m.save(output_path)
        print(f"✅ Map saved to {output_path}")

    return m

# --- 1️⃣5️⃣ Weighted K-means clustering ---
def weighted_kmeans(facilities, n_clusters):
    """Perform K-means clustering with frequency weights"""
    # Create weighted coordinates
    weighted_coords = []
    weights = []

    for _, row in facilities.iterrows():
        freq = int(row['frequency']) if 'frequency' in row else 1
        weighted_coords.extend([[row['latitude'], row['longitude']]] * freq)
        weights.extend([1] * freq)

    weighted_coords = np.array(weighted_coords)

    # Perform clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(weighted_coords)

    return kmeans.cluster_centers_

# --- 1️⃣6️⃣ Export all results ---
def export_all_results(scenarios, coverage_stats, output_folder):
    """Export all results to files"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Create summary dataframe
    summary_data = []
    for n_wh, data in scenarios.items():
        summary_data.append({
            'Warehouses': n_wh,
            'Selected_Warehouses': ', '.join(data['warehouses']),
            'Total_Distance_km': data['total_dist'],
            'Avg_Distance_km': data['avg_dist'],
            'Max_Distance_km': data['max_dist'],
            'Total_Cost': data['total_cost'],
            'Facilities_Served': len(data['assignments'])
        })

    summary_df = pd.DataFrame(summary_data)

    # Save summary
    summary_path = f"{output_folder}/scenario_summary_{timestamp}.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"✅ Summary saved to {summary_path}")

    # Save detailed assignments for each scenario
    for n_wh, data in scenarios.items():
        assignments_path = f"{output_folder}/assignments_{n_wh}wh_{timestamp}.csv"
        data['assignments'].to_csv(assignments_path, index=False)
        print(f"✅ Assignments for {n_wh} warehouses saved to {assignments_path}")

    # Save coverage analysis
    coverage_data = []
    for n_wh, stats in coverage_stats.items():
        for threshold, cov in stats['coverage_by_threshold'].items():
            coverage_data.append({
                'Warehouses': n_wh,
                'Threshold': threshold,
                'Facilities_Covered': cov['count'],
                'Coverage_Percentage': cov['percentage']
            })

    coverage_df = pd.DataFrame(coverage_data)
    coverage_path = f"{output_folder}/coverage_analysis_{timestamp}.csv"
    coverage_df.to_csv(coverage_path, index=False)
    print(f"✅ Coverage analysis saved to {coverage_path}")

    return summary_df

# --- 1️⃣7️⃣ Main execution ---
print("\n" + "="*50)
print("🚀 STARTING SUPPLY CHAIN OPTIMIZATION")
print("="*50 + "\n")

# Scenario parameters
scenario_warehouses = [1, 3, 5, 7, 10]  # number of regional warehouses
scenarios = {}

# Run scenarios
for n_wh in scenario_warehouses:
    print(f"\n{'='*40}")
    print(f"📦 Running scenario: {n_wh} regional warehouses")
    print(f"{'='*40}")

    # Weighted K-means clustering
    print("🔄 Performing weighted clustering...")
    centroids = weighted_kmeans(df_facilities, n_wh)

    # Select nearest cities to centroids
    selected_wh = []
    for lat, lon in centroids:
        distances = ((df_cities['latitude'] - lat)**2 + (df_cities['longitude'] - lon)**2)
        nearest_city = df_cities.iloc[distances.argmin()]['city']
        if nearest_city not in selected_wh:
            selected_wh.append(nearest_city)

    # If we don't have enough unique cities, add more
    while len(selected_wh) < n_wh:
        remaining = df_cities[~df_cities['city'].isin(selected_wh)]
        if len(remaining) > 0:
            selected_wh.append(remaining.iloc[0]['city'])
        else:
            break

    wh_df = df_cities[df_cities['city'].isin(selected_wh)]
    print(f"✅ Selected warehouses: {', '.join(selected_wh)}")

    # Assign facilities
    result_df = assign_facilities_parallel(df_facilities, wh_df, ors_cache, use_parallel=True)

    # Calculate metrics
    total_dist = result_df['distance_km'].sum()
    avg_dist = result_df['distance_km'].mean()
    total_cost = result_df['transport_cost'].sum()
    max_dist = result_df['distance_km'].max()

    # Store scenario results
    scenarios[n_wh] = {
        'warehouses': selected_wh,
        'total_dist': total_dist,
        'avg_dist': avg_dist,
        'total_cost': total_cost,
        'max_dist': max_dist,
        'assignments': result_df
    }

    print(f"\n📊 Results for {n_wh} warehouses:")
    print(f"   Total distance: {total_dist:,.2f} km")
    print(f"   Average distance: {avg_dist:.2f} km")
    print(f"   Total cost: ${total_cost:,.2f}")
    print(f"   Maximum distance: {max_dist:.2f} km")

# Save cache
ors_cache.save_cache()
print(f"\n✅ Cache statistics: {ors_cache.get_stats()}")

# --- 1️⃣8️⃣ Coverage analysis ---
print("\n📊 Performing coverage analysis...")
coverage_stats = analyze_coverage(scenarios, thresholds=[50, 100, 200, 500])

# Display coverage results
for n_wh, stats in coverage_stats.items():
    print(f"\n📦 {n_wh} Warehouses Coverage:")
    for threshold, cov in stats['coverage_by_threshold'].items():
        print(f"   {threshold}: {cov['count']} facilities ({cov['percentage']:.1f}%)")

# --- 1️⃣9️⃣ Detailed cost analysis ---
print("\n💰 Performing detailed cost analysis...")
cost_analyses = {}
for n_wh, data in scenarios.items():
    cost_analyses[n_wh] = calculate_detailed_costs(
        data,
        fuel_cost_per_km=0.5,
        vehicle_capacity=1000,
        driver_cost_per_hour=10,
        maintenance_per_km=0.1
    )

    print(f"\n📦 {n_wh} Warehouses - Cost Breakdown:")
    summary = cost_analyses[n_wh]['summary']
    print(f"   Fuel cost: ${summary['fuel_cost']:,.2f}")
    print(f"   Driver cost: ${summary['driver_cost']:,.2f}")
    print(f"   Maintenance: ${summary['maintenance_cost']:,.2f}")
    print(f"   Total cost: ${summary['total_cost']:,.2f}")
    print(f"   Cost per km: ${summary['cost_per_km']:.2f}")

# --- 2️⃣0️⃣ Visualizations ---
print("\n📈 Creating visualizations...")

# Create summary dataframe
summary_df = pd.DataFrame({
    'Warehouses': list(scenarios.keys()),
    'Total Distance (km)': [scenarios[n]['total_dist'] for n in scenarios],
    'Avg Distance (km)': [scenarios[n]['avg_dist'] for n in scenarios],
    'Total Cost ($)': [scenarios[n]['total_cost'] for n in scenarios],
    'Max Distance (km)': [scenarios[n]['max_dist'] for n in scenarios]
})

# 1. Scenario comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ['Total Distance (km)', 'Avg Distance (km)', 'Total Cost ($)', 'Max Distance (km)']

for ax, metric in zip(axes.flat, metrics):
    ax.plot(summary_df['Warehouses'], summary_df[metric], 'bo-', linewidth=2, markersize=8)
    ax.set_xlabel('Number of Warehouses', fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f'{metric} vs Warehouse Count', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # Add value labels
    for i, (x, y) in enumerate(zip(summary_df['Warehouses'], summary_df[metric])):
        ax.annotate(f'{y:,.0f}', (x, y), textcoords="offset points",
                   xytext=(0,10), ha='center', fontsize=9)

plt.tight_layout()
plot_path = f"{OUTPUT_FOLDER}/scenario_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved to {plot_path}")

# 2. Coverage heatmap
coverage_df = pd.DataFrame({
    'Warehouses': [n_wh for n_wh in scenarios for th in coverage_stats[n_wh]['coverage_by_threshold']],
    'Threshold': [th for n_wh in scenarios for th in coverage_stats[n_wh]['coverage_by_threshold']],
    'Coverage %': [cov['percentage'] for n_wh in scenarios
                   for cov in coverage_stats[n_wh]['coverage_by_threshold'].values()]
})

pivot_coverage = coverage_df.pivot(index='Warehouses', columns='Threshold', values='Coverage %')

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_coverage, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Coverage %'})
plt.title('Facility Coverage by Distance Threshold', fontsize=14, fontweight='bold')
plt.xlabel('Distance Threshold', fontsize=12)
plt.ylabel('Number of Warehouses', fontsize=12)
plt.tight_layout()
heatmap_path = f"{OUTPUT_FOLDER}/coverage_heatmap_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Heatmap saved to {heatmap_path}")

# 3. Distribution plot for best scenario
best_scenario = max(scenarios.keys(), key=lambda x: scenarios[x]['avg_dist'] < scenarios[min(scenarios.keys())]['avg_dist'])

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(scenarios[best_scenario]['assignments']['distance_km'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Distance (km)', fontsize=11)
plt.ylabel('Number of Facilities', fontsize=11)
plt.title(f'Distance Distribution ({best_scenario} Warehouses)', fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
facilities_per_wh = scenarios[best_scenario]['assignments']['warehouse_city'].value_counts()
plt.bar(range(len(facilities_per_wh)), facilities_per_wh.values)
plt.xlabel('Warehouse', fontsize=11)
plt.ylabel('Facilities Served', fontsize=11)
plt.title(f'Facilities per Warehouse ({best_scenario} Warehouses)', fontsize=12, fontweight='bold')
plt.xticks(range(len(facilities_per_wh)), [f'W{i+1}' for i in range(len(facilities_per_wh))])
plt.grid(True, alpha=0.3)

plt.tight_layout()
dist_path = f"{OUTPUT_FOLDER}/distribution_{best_scenario}wh_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
plt.savefig(dist_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Distribution plot saved to {dist_path}")

# 4. Create interactive map for best scenario
print("\n🗺️ Creating interactive map for best scenario...")
map_path = f"{OUTPUT_FOLDER}/interactive_map_{best_scenario}wh_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"
interactive_map = create_interactive_map(scenarios[best_scenario], best_scenario, map_path)

# Display in Colab
from IPython.display import IFrame
display(IFrame(map_path, width=800, height=600))

# --- 2️⃣1️⃣ Route optimization for best scenario (optional) ---
if ORTOOLS_AVAILABLE:
    print("\n🚚 Performing route optimization for best scenario...")
    route_optimizations = {}

    for warehouse in scenarios[best_scenario]['warehouses'][:2]:  # Limit to first 2 warehouses for demo
        print(f"\n🔄 Optimizing routes for {warehouse}...")

        # Get facilities for this warehouse
        wh_facilities = scenarios[best_scenario]['assignments'][
            scenarios[best_scenario]['assignments']['warehouse_city'] == warehouse
        ]

        # Get warehouse location
        wh_location = df_cities[df_cities['city'] == warehouse].iloc[0]

        # Optimize routes
        routes = optimize_routes(
            wh_location,
            wh_facilities.head(20),  # Limit to 20 facilities for demo
            num_vehicles=3,
            ors_cache=ors_cache
        )

        if routes:
            route_optimizations[warehouse] = routes
            print(f"✅ Optimized {len(routes)} routes for {warehouse}")
            for route in routes:
                print(f"   Vehicle {route['vehicle_id']}: {route['stops']} stops, {route['distance_km']:.1f} km")

# --- 2️⃣2️⃣ Export all results ---
print("\n💾 Exporting all results...")
summary_df = export_all_results(scenarios, coverage_stats, OUTPUT_FOLDER)

# --- 2️⃣3️⃣ Final summary ---
print("\n" + "="*50)
print("✅ SUPPLY CHAIN OPTIMIZATION COMPLETE")
print("="*50)
print(f"\n📊 Best performing configuration: {best_scenario} warehouses")
print(f"   Average distance: {scenarios[best_scenario]['avg_dist']:.2f} km")
print(f"   Total cost: ${scenarios[best_scenario]['total_cost']:,.2f}")
print(f"\n📁 All outputs saved to: {OUTPUT_FOLDER}")
print("\n🏁 Done!")

Setting up environment...

Installing core packages...
Installing OR-Tools...
Installing visualization packages...
Installing geospatial packages...
Installing scikit-learn...
Installing OpenRouteService...

✅ Package installation complete!
✅ OpenRouteService imported successfully
✅ OR-Tools imported successfully
✅ All imports complete!

📦 Protobuf version: 3.20.3
Note: Some warnings about incompatible versions are expected and usually harmless.
The packages will still work as long as core functionality isn't broken.

⚠️  No API key found in secrets.
✅ Google Drive already mounted
✅ Output folder: /content/drive/MyDrive/supply_chain_outputs/

📊 Loading facilities data...
✅ Loaded 19693 facilities across 37 states
📈 Facility types: {'Primary Health Centre': 4359, 'Clinic': 4126, 'Health Centre': 3221, 'Dispensary': 3075, 'Health Post': 2880, 'Basic Health Centre': 529, 'General Hospital': 526, 'Comprehensive Health Centre': 408, 'Cottage Hospital': 148, 'Hospital': 144, 'Model Health Ce

Calculating distances:   4%|▍         | 821/19693 [30:42<1:44:41,  3.00it/s]